In [1]:
import os
import contextlib
from pathlib import Path
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from tqdm import tqdm


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [3]:
DATA_ROOT = Path("/home/n.bezborodov/ai/lab2/data")

train_img_dir = DATA_ROOT / "tiff" / "train"
train_mask_dir = DATA_ROOT / "tiff" / "train_labels"

val_img_dir = DATA_ROOT / "tiff" / "val"
val_mask_dir = DATA_ROOT / "tiff" / "val_labels"

test_img_dir = DATA_ROOT / "tiff" / "test"
test_mask_dir = DATA_ROOT / "tiff" / "test_labels"

print("train:", len(list(train_img_dir.glob("*"))), len(list(train_mask_dir.glob("*"))))
print("val  :", len(list(val_img_dir.glob("*"))), len(list(val_mask_dir.glob("*"))))
print("test :", len(list(test_img_dir.glob("*"))), len(list(test_mask_dir.glob("*"))))

train: 1108 1108
val  : 14 14
test : 49 49


In [4]:

TILE_SIZE = 512
STRIDE = 256

BATCH_SIZE = 6
NUM_WORKERS = 4

LR = 3e-4
NUM_EPOCHS = 40

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


In [5]:
def generate_positions(size, tile_size, stride):
    positions = list(range(0, size - tile_size + 1, stride))
    if not positions:
        return [0]
    if positions[-1] != size - tile_size:
        positions.append(size - tile_size)
    return positions

In [6]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(),
    ToTensorV2(),
])

In [7]:
@contextlib.contextmanager
def suppress_stderr():
    old_stderr_fd = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    os.close(devnull)
    try:
        yield
    finally:
        os.dup2(old_stderr_fd, 2)
        os.close(old_stderr_fd)

In [8]:
class RoadsWindowDataset(Dataset):
    def __init__(self, images_dir, masks_dir, tile_size=512, stride=256, transform=None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.tile_size = tile_size
        self.stride = stride
        self.transform = transform

        self.image_paths = sorted(self.images_dir.glob("*"))
        self.mask_paths = sorted(self.masks_dir.glob("*"))

        assert len(self.image_paths) == len(self.mask_paths), "images/masks count mismatch"

        self.samples = []

        for img_path, mask_path in zip(self.image_paths, self.mask_paths):
            mask = tiff.imread(str(mask_path))

            if mask.ndim == 3:
                mask = mask[..., 0]

            h, w = mask.shape[:2]
            ys = generate_positions(h, self.tile_size, self.stride)
            xs = generate_positions(w, self.tile_size, self.stride)

            for y in ys:
                for x in xs:
                    self.samples.append((img_path, mask_path, x, y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, x, y = self.samples[idx]

        image = tiff.imread(str(img_path))
        mask = tiff.imread(str(mask_path))

        if image.ndim == 3 and image.shape[0] in [3, 4] and image.shape[-1] not in [3, 4]:
            image = np.transpose(image, (1, 2, 0))

        if mask.ndim == 3:
            mask = mask[..., 0]

        image = image[y:y + self.tile_size, x:x + self.tile_size]
        mask = mask[y:y + self.tile_size, x:x + self.tile_size]

        image = image.astype(np.uint8)
        mask = (mask > 0).astype(np.float32)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        return image, mask.unsqueeze(0)

In [9]:
train_dataset = RoadsWindowDataset(
    images_dir=train_img_dir,
    masks_dir=train_mask_dir,
    tile_size=512,
    stride=256,
    transform=train_transform
)

val_dataset = RoadsWindowDataset(
    images_dir=val_img_dir,
    masks_dir=val_mask_dir,
    tile_size=512,
    stride=256,
    transform=val_transform
)

test_dataset = RoadsWindowDataset(
    images_dir=test_img_dir,
    masks_dir=test_mask_dir,
    tile_size=512,
    stride=256,
    transform=val_transform
)

print("train windows:", len(train_dataset))
print("val windows  :", len(val_dataset))
print("test windows :", len(test_dataset))

NameError: name 'tiff' is not defined

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [ ]:
images, masks = next(iter(train_loader))
print(images.shape, masks.shape)

img = images[0].permute(1, 2, 0).cpu().numpy()
msk = masks[0, 0].cpu().numpy()

img = (img - img.min()) / (img.max() - img.min() + 1e-8)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("train tile")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(msk, cmap="gray")
plt.title("train mask")
plt.axis("off")

plt.show()

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2, 3))
    den = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + eps
    dice = num / den
    return 1 - dice.mean()

def total_loss(logits, targets):
    return 0.5 * bce_loss(logits, targets) + 0.5 * dice_loss(logits, targets)

def iou_score(logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    intersection = (preds * targets).sum(dim=(2, 3))
    union = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


def dice_score(logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    intersection = (preds * targets).sum(dim=(2, 3))
    total = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))

    dice = (2 * intersection + eps) / (total + eps)
    return dice.mean().item()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

In [ ]:
writer = SummaryWriter(log_dir="runs/roads_unet")

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss_value = 0.0
    total_iou = 0.0
    total_dice = 0.0

    for images, masks in tqdm(loader, desc="train", leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = total_loss(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss_value += loss.item()

        with torch.no_grad():
            total_iou += iou_score(logits, masks)
            total_dice += dice_score(logits, masks)

    return (
        total_loss_value / len(loader),
        total_iou / len(loader),
        total_dice / len(loader),
    )


@torch.no_grad()
def validate_one_epoch(model, loader, device):
    model.eval()

    total_loss_value = 0.0
    total_iou = 0.0
    total_dice = 0.0

    for images, masks in tqdm(loader, desc="val", leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(images)
        loss = total_loss(logits, masks)

        total_loss_value += loss.item()
        total_iou += iou_score(logits, masks)
        total_dice += dice_score(logits, masks)

    return (
        total_loss_value / len(loader),
        total_iou / len(loader),
        total_dice / len(loader),
    )

In [ ]:
@torch.no_grad()
def log_predictions_to_tensorboard(model, loader, device, writer, epoch, num_images=3):
    model.eval()

    images, masks = next(iter(loader))
    images = images.to(device)

    logits = model(images)
    preds = (torch.sigmoid(logits) > 0.5).float().cpu()

    for i in range(min(num_images, images.size(0))):
        img = images[i].cpu()
        true_mask = masks[i].cpu()
        pred_mask = preds[i].cpu()

        writer.add_image(f"Image/{i}", img, epoch)
        writer.add_image(f"TrueMask/{i}", true_mask, epoch)
        writer.add_image(f"PredMask/{i}", pred_mask, epoch)

In [ ]:
writer = SummaryWriter("runs/roads_unet")
best_iou = 0.0
history = []

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    train_loss, train_iou, train_dice = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss, val_iou, val_dice = validate_one_epoch(model, val_loader, DEVICE)

    scheduler.step(val_iou)

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("IoU/train", train_iou, epoch)
    writer.add_scalar("Dice/train", train_dice, epoch)

    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("IoU/val", val_iou, epoch)
    writer.add_scalar("Dice/val", val_dice, epoch)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_iou": train_iou,
        "train_dice": train_dice,
        "val_loss": val_loss,
        "val_iou": val_iou,
        "val_dice": val_dice,
    })

    print(
        f"epoch {epoch+1:02d}: "
        f"train_loss={train_loss:.4f} "
        f"train_iou={train_iou:.4f} "
        f"train_dice={train_dice:.4f} "
        f"val_loss={val_loss:.4f} "
        f"val_iou={val_iou:.4f} "
        f"val_dice={val_dice:.4f}"
    )

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "best_roads_unet.pth")
        print("saved best model")

writer.close()

history_df = pd.DataFrame(history)
history_df

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
plt.legend()
plt.title("Loss")

plt.subplot(1, 3, 2)
plt.plot(history_df["epoch"], history_df["val_iou"], label="val_iou")
plt.legend()
plt.title("IoU")

plt.subplot(1, 3, 3)
plt.plot(history_df["epoch"], history_df["val_dice"], label="val_dice")
plt.legend()
plt.title("Dice")

plt.show()

In [10]:
best_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=1
)

best_model.load_state_dict(torch.load("best_roads_unet.pth", map_location=DEVICE))
best_model = best_model.to(DEVICE)
best_model.eval()

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [11]:
import torch
import numpy as np
from tqdm import tqdm

@torch.no_grad()
def evaluate_thresholds(model, loader, device, thresholds=None, eps=1e-7):
    model.eval()

    if thresholds is None:
        thresholds = np.arange(0.2, 0.81, 0.05)

    intersections = {float(t): 0.0 for t in thresholds}
    unions = {float(t): 0.0 for t in thresholds}

    for images, masks in tqdm(loader, desc="threshold search", leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        probs = torch.sigmoid(model(images))

        for t in thresholds:
            preds = (probs > t).float()
            intersection = (preds * masks).sum().item()
            union = (preds.sum() + masks.sum() - intersection)

            intersections[float(t)] += intersection
            unions[float(t)] += union

    results = []
    for t in thresholds:
        t = float(t)
        iou = (intersections[t] + eps) / (unions[t] + eps)
        results.append((t, iou))

    results.sort(key=lambda x: x[1], reverse=True)
    return results

In [12]:
threshold_results = evaluate_thresholds(model, val_loader, DEVICE)

for t, iou in threshold_results:
    print(f"threshold={t:.2f}, val_iou={iou:.4f}")

NameError: name 'model' is not defined

In [13]:
best_threshold, best_val_iou = threshold_results[0]
print("best_threshold =", best_threshold)
print("best_val_iou   =", best_val_iou)

NameError: name 'threshold_results' is not defined

In [14]:
test_loss, test_iou, test_dice = validate_one_epoch(model, test_loader, DEVICE)

print(f"test_loss = {test_loss:.4f}")
print(f"test_iou  = {test_iou:.4f}")
print(f"test_dice = {test_dice:.4f}")

NameError: name 'validate_one_epoch' is not defined

In [15]:
@torch.no_grad()
def show_predictions(model, loader, device, n=3):
    model.eval()

    images, masks = next(iter(loader))
    images = images.to(device)

    logits = model(images)
    preds = (torch.sigmoid(logits) > 0.5).float().cpu()

    for i in range(min(n, images.size(0))):
        img = images[i].cpu().permute(1, 2, 0).numpy()
        true_mask = masks[i, 0].cpu().numpy()
        pred_mask = preds[i, 0].numpy()

        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(true_mask, cmap="gray")
        plt.title("True mask")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(pred_mask, cmap="gray")
        plt.title("Pred mask")
        plt.axis("off")

        plt.show()

In [16]:
show_predictions(model, test_loader, DEVICE, n=5)

NameError: name 'model' is not defined